# Lab 3: Data Cleaning and Transformation Pipeline
## Carlos Muñiz Lara

In [1]:
from spark_utils import SparkUtils
su = SparkUtils()
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/24 00:39:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkContext master=spark://spark-master:7077 appName=helloworld>

## Spark data sources
 df = spark.read.format("").option("mode","FAILFAST").option("schema", <schema created>. option("header", "true").option("path", always pass folder not specific file))

In [2]:
columns_types = [("Order_ID", "long"),
                ("Company", "string"),
                ("City", "string"),
                ("Customer_Age", "int"),
                ("Order_Value", "float"),
                ("Delivery_Time_Min", "float"),
                ("Distance_Km", "float"),
                ("Items_Count", "float"),
                ("Product_Category", "string"),
                ("Payment_Method", "string"),
                ("Customer_Rating", "float"),
                ("Discount_Applied", "float"),
                ("Delivery_Partner_Rating", "float"),
                ]



In [3]:
qcommerce_schema = SparkUtils.generate_schema(columns_types)

qcommerce_df = su._spark \
                .read \
                .option("header", "true") \
                .schema(qcommerce_schema) \
                .csv("/opt/spark/work-dir/data/")

qcommerce_df.printSchema()

root
 |-- Order_ID: long (nullable = true)
 |-- Company: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer_Age: integer (nullable = true)
 |-- Order_Value: float (nullable = true)
 |-- Delivery_Time_Min: float (nullable = true)
 |-- Distance_Km: float (nullable = true)
 |-- Items_Count: float (nullable = true)
 |-- Product_Category: string (nullable = true)
 |-- Payment_Method: string (nullable = true)
 |-- Customer_Rating: float (nullable = true)
 |-- Discount_Applied: float (nullable = true)
 |-- Delivery_Partner_Rating: float (nullable = true)



In [5]:
qcommerce_df.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.0|                    3.2|
| 1000003|Flipkart Minute

In [6]:
qcommerce_null_count_df = SparkUtils.count_nulls(qcommerce_df)
qcommerce_null_count_df.show()

+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|Order_ID|Company| City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+
|       0|      0|52000|           0|          0|                0|          0|      35000|               0|             0|          47000|               0|                 104137|
+--------+-------+-----+------------+-----------+-----------------+-----------+-----------+----------------+--------------+---------------+----------------+-----------------------+



In [7]:
qcommerce_no_nulls_df = qcommerce_df.na.drop()
print(f"size of data after dropping nulls: {qcommerce_no_nulls_df.count()}")

size of data after dropping nulls: 780992


### Instead of removing the df we are going to fill those collumns

In [9]:
qcommerce_wo_nulls_fillna = qcommerce_df.fillna({
    'City': 'Unknown',
    'Items_Count':0.0,
    'Customer_Rating': 0.0,
    'Delivery_Partner_Rating': 0.0
})

print(f"size of data after filling nulls: {qcommerce_wo_nulls_fillna.count()}")

size of data after filling nulls: 1000000


In [11]:
from pyspark.sql.functions import lit
qcommerce_wo_nulls_fillna = qcommerce_wo_nulls_fillna.withColumn("Code_Country", lit("India"))
qcommerce_wo_nulls_fillna.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|       India|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74|       10.0|          Snacks|Cash on Delivery|            2.3|             0.

In [13]:
from pyspark.sql.functions import col
qcommerce_wo_nulls_fillna_tax = qcommerce_wo_nulls_fillna.withColumn("Tax", col("Order_Value") * 0.18)
qcommerce_wo_nulls_fillna_tax.show(5)

+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+------------------+
|Order_ID|         Company|    City|Customer_Age|Order_Value|Delivery_Time_Min|Distance_Km|Items_Count|Product_Category|  Payment_Method|Customer_Rating|Discount_Applied|Delivery_Partner_Rating|Code_Country|               Tax|
+--------+----------------+--------+------------+-----------+-----------------+-----------+-----------+----------------+----------------+---------------+----------------+-----------------------+------------+------------------+
| 1000001|Swiggy Instamart|   Noida|          46|   702.3375|           19.182|      11.97|       12.0|           Dairy|          Wallet|            2.1|             1.0|                    3.2|       India|126.42075439453124|
| 1000002|Flipkart Minutes|Amritsar|          56|     1007.3|           19.644|      12.74| 

## Task 1

In [18]:
from pyspark.sql.functions import when
qcommerce_task1=qcommerce_wo_nulls_fillna_tax.withColumn("Delivery_SLA", when(col("Delivery_Time_Min")<=15, "Fast").when((col("Delivery_Time_Min")>15) & (col("Delivery_Time_Min")<=20), "On Time").otherwise("Late")).filter(col("Delivery_SLA")=="Late").orderBy(col("Delivery_Time_Min").desc())

qcommerce_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()

+--------+--------+--------+-----------------+------------+
|Order_ID| Company|    City|Delivery_Time_Min|Delivery_SLA|
+--------+--------+--------+-----------------+------------+
| 1529779|Jio Mart|Haridwar|             40.0|        Late|
| 1807126|Jio Mart|Haridwar|             40.0|        Late|
| 1165764|Jio Mart|Haridwar|           39.994|        Late|
| 1610720|Jio Mart|Haridwar|           39.994|        Late|
| 1729503|Jio Mart|Haridwar|           39.994|        Late|
| 1951122|Jio Mart|Haridwar|           39.988|        Late|
| 1975896|Jio Mart|Haridwar|           39.988|        Late|
| 1059671|Jio Mart|Haridwar|           39.982|        Late|
| 1594835|Jio Mart|Haridwar|           39.982|        Late|
| 1162804|Jio Mart|Haridwar|           39.982|        Late|
| 1826295|Jio Mart|Haridwar|           39.982|        Late|
| 1961544|Jio Mart|Haridwar|           39.976|        Late|
| 1360875|Jio Mart|Haridwar|           39.964|        Late|
| 1555058|Jio Mart|Haridwar|           3

## Task 2

In [17]:
qcommerce_task1.select("Order_ID", "Company", "City", "Delivery_Time_Min", "Delivery_SLA").show()

+--------+----------------+---------+-----------------+------------+
|Order_ID|         Company|     City|Delivery_Time_Min|Delivery_SLA|
+--------+----------------+---------+-----------------+------------+
| 1000001|Swiggy Instamart|    Noida|           19.182|     On Time|
| 1000002|Flipkart Minutes| Amritsar|           19.644|     On Time|
| 1000003|Flipkart Minutes|   Mumbai|            16.91|     On Time|
| 1000004|Swiggy Instamart|    Delhi|            5.864|        Fast|
| 1000005|           Dunzo|   Mumbai|            12.47|        Fast|
| 1000006|        Jio Mart|  Kolkata|           19.738|     On Time|
| 1000007|         Blinkit| Bengluru|           18.476|     On Time|
| 1000008|           Dunzo|  Chennai|           10.916|        Fast|
| 1000009|Flipkart Minutes| Bengluru|           18.174|     On Time|
| 1000010|Swiggy Instamart|  Chennai|           11.238|        Fast|
| 1000011|         Blinkit|Hyderabad|           12.838|        Fast|
| 1000012|Swiggy Instamart|     Pu

In [21]:
from pyspark.sql.functions import count, avg
qcommerce_task2 = qcommerce_wo_nulls_fillna_tax.withColumn("Effective_Order_Value",col("Order_Value")*(1-col("Discount_Applied")))
qcommerce_task2 = qcommerce_task2.withColumn("Price_Bucket", when(col("Effective_Order_Value")<200, "Low").when((col("Effective_Order_Value")>=200) & (col("Effective_Order_Value")<500), "Medium").otherwise("High"))
qcommerce_task2= qcommerce_task2.groupBy("Price_Bucket").agg(count("*").alias("Count"),avg("Effective_Order_Value").alias("Avg_Effective_Order_Value")).orderBy(col("Avg_Effective_Order_Value").desc()).show()


+------------+------+-------------------------+
|Price_Bucket| Count|Avg_Effective_Order_Value|
+------------+------+-------------------------+
|        High|268954|        740.4328299308322|
|      Medium|210428|       358.09664898740635|
|         Low|520618|       21.591204157544553|
+------------+------+-------------------------+



## Task 3

In [23]:
task3 = qcommerce_wo_nulls_fillna_tax.withColumn("Age_Group", when(col("Customer_Age")<25, "Young").when((col("Customer_Age")>=25) & (col("Customer_Age")<44), "Adult").otherwise("Senior")).filter((col("Customer_Age")>0) & (col("Customer_Age")<100)).groupBy("Age_Group","Product_Category").agg((count("*").alias("Orders")),avg("Order_Value").alias("Average_Order_Value")).orderBy((col("Age_Group").asc()),col("Orders").desc()).show()

+---------+-------------------+------+-------------------+
|Age_Group|   Product_Category|Orders|Average_Order_Value|
+---------+-------------------+------+-------------------+
|    Adult|              Dairy| 65023|   573.399541336331|
|    Adult|      Personal Care| 65013|  569.5898788289264|
|    Adult|          Groceries| 64968|  571.7178878346155|
|    Adult|             Snacks| 64685|  570.5328394703844|
|    Adult|          Household| 64678|  572.8577889980605|
|    Adult|          Beverages| 64666|   572.099575839277|
|    Adult|Fruits & Vegetables| 64390|  569.7405814283252|
|   Senior|              Dairy| 54514|  573.7910668086589|
|   Senior|          Groceries| 54353|  571.9842917366681|
|   Senior|             Snacks| 54339|  572.3606182995497|
|   Senior|          Household| 54235|  571.3643088361069|
|   Senior|          Beverages| 54059|  568.2998789788686|
|   Senior|      Personal Care| 54025|  570.9699125884629|
|   Senior|Fruits & Vegetables| 53988|  573.626560951680

In [24]:
su._spark.stop()